# Run Full Benchmark 4 Evaluation & Comparison Pipeline

Runs the complete benchmark-4 permutation study end-to-end:

1. **Phase 1** — verify the mean-token embedding columns (`emb_mean_blocks_*`) exist in the
   all-layers parquet files; if missing, run `stage_1_extraction/extract_benchmark4_embeddings.py`
   (the merged extraction script; its earlier steps no-op fast when outputs already exist).
2. **Phase 2** — train all 4 model permutations over all 32 Evo2 layers via the merged trainer
   `stage_5_benchmark4_training/train_benchmark4_permutations.py`:
   - `--experiment xgb_block` (XGBoost, last token)
   - `--experiment xgb_mean` (XGBoost, mean token)
   - `--experiment mlp_last` (MLP, last token)
   - `--experiment mlp_mean` (MLP, mean token)
3. **Phase 3** — `stage_6_plots/plot_all_permutations.py`: per-model accuracy-by-layer bar charts,
   the combined comparison plot, and the summary comparison table.

Every step is a standalone script under `notebooks/stage_*/` (resumable: already-trained layers /
existing `results_summary.json` are skipped automatically), launched here via subprocess so the
full pipeline is reproducible with a single top-to-bottom run. Shared helpers live in
`notebooks/definitions.py`.

**Note:** All paths resolve relative to the project root (the directory containing `data/`,
`models/` and `notebooks/`), so the notebook works from any working directory inside the
finalized codebase. There are no hardcoded paths. Set `CUDA_DEVICE` below if you need to
pin a specific GPU (e.g. `"1"`).


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path


def _find_project_root():
    """Locate <root> — the directory containing data/, models/ and notebooks/."""
    starts = []
    if "__file__" in globals() and __file__:  # if exported from a notebook as a .py
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd())
    for start in starts:
        for cand in (start, *start.parents):
            if (cand / "data").is_dir() and (cand / "models").is_dir() and (cand / "notebooks").is_dir():
                return cand
    raise RuntimeError(
        "Could not locate the project root (a directory containing data/, models/ and notebooks/). "
        "Run the notebook from any working directory inside the finalized codebase."
    )


PROJECT_ROOT  = _find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR      = PROJECT_ROOT / "data"
MODELS_DIR    = PROJECT_ROOT / "models"

# Pin a specific GPU if needed (None = let the training scripts pick the default).
# e.g. CUDA_DEVICE = "1" when GPU 0 is busy.
CUDA_DEVICE = None

print(f"Project root : {PROJECT_ROOT}")
print(f"Notebooks dir: {NOTEBOOKS_DIR}")

In [ ]:
def run_step(name, script_rel, extra_args=()):
    """Run one pipeline script (subprocess, streamed), return (ok, seconds).

    script_rel is relative to notebooks/ (e.g. "stage_5_benchmark4_training/train_benchmark4_permutations.py").
    """
    script = NOTEBOOKS_DIR / script_rel
    if not script.is_file():
        raise FileNotFoundError(f"Missing pipeline script: {script}")
    env = dict(os.environ)
    if CUDA_DEVICE:
        env["CUDA_VISIBLE_DEVICES"] = CUDA_DEVICE
    print()
    print("=" * 70)
    print(f"STEP: {name}  ->  {script_rel} {' '.join(extra_args)}".strip() +
          (f"  (CUDA_VISIBLE_DEVICES={CUDA_DEVICE})" if CUDA_DEVICE else ""))
    print("=" * 70, flush=True)
    t0 = time.time()
    proc = subprocess.run([sys.executable, str(script), *extra_args], env=env, cwd=str(PROJECT_ROOT))
    dt = time.time() - t0
    ok = proc.returncode == 0
    print(f"{'OK ' if ok else 'FAIL'} {name}: {dt:.0f}s (exit code {proc.returncode})", flush=True)
    return ok, dt


step_results = []  # (name, ok, seconds)

In [ ]:
# ==== PHASE 1: Ensure mean-token embedding columns exist ====
TRAIN_PQ = DATA_DIR / "benchmark_4" / "Train_all_layers_embeddings.parquet"
if not TRAIN_PQ.is_file():
    raise FileNotFoundError(
        f"Missing {TRAIN_PQ}. Run the benchmark-4 extraction script first "
        "(stage_1_extraction/extract_benchmark4_embeddings.py)."
    )

import pandas as pd
try:
    _tmp = pd.read_parquet(TRAIN_PQ, columns=["emb_mean_blocks_0"])
    has_mean = bool(_tmp["emb_mean_blocks_0"].notna().any())
    del _tmp
except Exception:
    has_mean = False

if has_mean:
    print("Mean-token columns already present. Skipping extraction.")
else:
    print("Mean-token columns not found. Running benchmark-4 extraction...", flush=True)
    ok, dt = run_step("Extract benchmark-4 embeddings", "stage_1_extraction/extract_benchmark4_embeddings.py")
    step_results.append(("Extract benchmark-4 embeddings", ok, dt))
    if not ok:
        raise RuntimeError("Benchmark-4 extraction failed — aborting pipeline.")

In [ ]:
# ==== PHASE 2: Train all 4 model permutations (32 layers each) ====
TRAINER = "stage_5_benchmark4_training/train_benchmark4_permutations.py"
EXPERIMENTS = [
    ("XGBoost, last token", "xgb_block"),
    ("XGBoost, mean token", "xgb_mean"),
    ("MLP, last token",     "mlp_last"),
    ("MLP, mean token",     "mlp_mean"),
]
for name, exp in EXPERIMENTS:
    ok, dt = run_step(f"Train {name}", TRAINER, ("--experiment", exp))
    step_results.append((f"Train {name}", ok, dt))

In [ ]:
# ==== PHASE 3: Comparison plots + summary table ====
ok, dt = run_step("Comparison plots & summary table", "stage_6_plots/plot_all_permutations.py")
step_results.append(("Comparison plots & summary table", ok, dt))

In [ ]:
# ==== SUMMARY ====
print()
print("=" * 62)
print(f"{'STEP':<40} {'STATUS':<6} {'TIME'}")
print("-" * 62)
for name, ok, dt in step_results:
    print(f"{name:<40} {'OK' if ok else 'FAIL':<6} {dt:.0f}s")
print("=" * 62)
failed = [n for n, ok, _ in step_results if not ok]
if failed:
    raise RuntimeError("Pipeline finished with failures: " + ", ".join(failed))
print("All results and plots are under:")
print(f"  models:  {MODELS_DIR / 'benchmark_4'}")
print(f"  figures: {PROJECT_ROOT / 'figures' / 'benchmark_4'}")
print("DONE!", flush=True)